In [1]:
# Experiment (Visualization) note on n-gram language modeling (notebook scale but with visualization)
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup
# For fast training, set 'BOS_TOKEN_ID' to 15 in 'sorl/gat_sim.py' 

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)


# 1. load a small subset of the dataset (checked)
# 2. tokenize & build data loader (checked)
# 3. train with SoRL. 
# 4. visualize abstraciton dynamics (similar to copy-n-paste, but more generally on per-n-gram statistics)

In [2]:
from data.tinystory_local import TinyStoriesDataLoader, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 16
K = 4
loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device='cpu')

doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
batch_size = 8
memory_span = 2 * max_len + 2
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 100 stories from TinyStories train...
Loaded 100 stories, 1600 tokens total, 0.01 MB
Collected 248 unique 4-chunks


In [3]:
# train SoRL 

# Dynamic Visualization of SoRL
# 1. select-one SoRL

from sorl.neo_utils import sorl_rollout_v2, select_best_per_doc, sorl_evaluate_v2, compute_vocab_utilization_rate
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss, SoRLLoss_v2, SoRLLoss_v3, SoRLLoss_v4

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

loss_fn = SoRLLoss_v4(model.vocab_sizes[1], model.vocab_sizes[0], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400 
alpha_abs = 0.1
# alpha_marg_ent = 1.0
alpha_soft_zipf = 1.0
alpha_traj_marg_ent = 1.0

record = defaultdict(list)
img_frames = []

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens, doc_ids = loader.get_batch(batch_size)

    with torch.no_grad(): 
        # --- breakdown of SoRL search (select one per-document) ---
        search_data, search_ppt = sorl_rollout_v2(tokens, model, n=n, K=K, 
                                    max_iterations=max_iterations,
                                    memory_span=memory_span,
                                    attn_blocksize=attn_blocksize,
                                    temperature=temperature,
                                    truncate_seq_len=False)
        search_ppt = search_ppt.reshape(search_data.shape[0], -1)

        levels = (search_data >= model.vocab_sizes[0]).long()
        best_data, best_ppt, best_ppt_advantage = select_best_per_doc(search_data, search_ppt, levels, model)

    # --- compute loss --- 
    traj_loss, abs_loss, zipf_bigram_loss, traj_marg_ent = loss_fn(best_data, model, memory_span, attn_blocksize)
    loss = traj_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss + alpha_traj_marg_ent * traj_marg_ent

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)
            val_tokens, val_adv, traj_loss, abs_loss, abs_logits, abs_tokens = sorl_evaluate_v2(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            _, _, zipf_bigram_loss, traj_marg_ent = loss_fn(val_tokens, model, memory_span, attn_blocksize)
            vocab_util = compute_vocab_utilization_rate(val_tokens, model)
            abs_stats.update(abs_logits, abs_tokens, doc_ids)
            
            record['vocab_util'].append(vocab_util * 100)
            record['search_adv'].append(val_adv.item() * 100)
            record['abs_loss'].append(abs_loss.item())
            record['traj_loss'].append(traj_loss.item())
            record['bigram_rep_rate'].append(abs_stats.bigram_rep_rate)
            record['kl_soft_zipf'].append(zipf_bigram_loss.item())
            # record['marg_ent'].append(marg_ent.item())
        print(f"\nvalidation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}% | vocab util: {vocab_util * 100:.2f}%  | kl_soft_zipf: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | traj_marg_ent: {traj_marg_ent.item():.2f}")
        
        img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        img_frames.append(img)
        # break



validation step 0 | traj_loss: 10.73 | abs_loss: 8.78 | search adv: 0.05% | vocab util: 56.25%  | kl_soft_zipf: 4.02 | bigram rep rate: 0.93 | traj_marg_ent: 10.72

validation step 2 | traj_loss: 10.63 | abs_loss: 8.29 | search adv: 0.15% | vocab util: 62.50%  | kl_soft_zipf: 2.77 | bigram rep rate: 0.87 | traj_marg_ent: 10.72

validation step 4 | traj_loss: 10.47 | abs_loss: 7.83 | search adv: 0.11% | vocab util: 50.00%  | kl_soft_zipf: 1.93 | bigram rep rate: 0.81 | traj_marg_ent: 10.72

validation step 6 | traj_loss: 10.19 | abs_loss: 6.97 | search adv: 0.24% | vocab util: 43.75%  | kl_soft_zipf: 1.54 | bigram rep rate: 0.74 | traj_marg_ent: 10.72

validation step 8 | traj_loss: 9.83 | abs_loss: 6.35 | search adv: 0.27% | vocab util: 25.00%  | kl_soft_zipf: 1.42 | bigram rep rate: 0.70 | traj_marg_ent: 10.72

validation step 10 | traj_loss: 9.37 | abs_loss: 5.78 | search adv: 0.35% | vocab util: 37.50%  | kl_soft_zipf: 1.33 | bigram rep rate: 0.63 | traj_marg_ent: 10.72

validation

In [ ]:
# (1). 

In [1]:
# Request #1. 
# -> build experiement pipeline to test out zipf regularization on FineWeb & TinyStories dataset

# Idea #1. 
# -> separatibility (contrast) can be induced by regularizing on MBE for abstract representations (WTE)
# -> I wonder what effect does this has -- can it enlarge search adv? 

# Idea #2. 
# -> an extra scalar reward r = max(p(s|a) - p(s), r_min) can create an extra 'rich gets richer' dynamics
#    that helps improve search adv?

# Idea #3. 
# -> explicitly regularize on p(s), try to push it up to see if we can further improve 'search advantage'


In [ ]:
# marg cond w = 1.0 
# traj_loss: 0.85 | abs_loss: 0.04 | search adv: 21.01% | vocab util: 93.75%  | marg_ent: 2.67 | cond_ent: 0.05

# ce zipf w = 1.0 
# traj_loss: 0.90 | abs_loss: 0.01 | search adv: 26.56% | vocab util: 6.25% (collapsed) | ce_zipf: 1.18 | ce_soft_zipf: 3.26

# kl zip w = 1.0 
# traj_loss: 1.21 | abs_loss: 0.28 | search adv: 31.68% | vocab util: 37.50%  | kl_zipf: 0.05 | kl_soft_zipf: 1.81
# (traj loss is still dropping for the record)

# soft kl zip w = 1.0
# traj_loss: 0.87 | abs_loss: 0.51 | search adv: 30.31% | vocab util: 37.5% - 56%  | kl_zipf: 0.04 | kl_soft_zipf: 0.12
# -> soft kl seems to be a better target

# soft bigram zipf kl w = 1.0 
# traj_loss: 0.89 | abs_loss: 0.53 | search adv: 25.84% | vocab util: 68.75%  | kl_soft_zipf: 0.24
# -> visually I observe much less 'repetitions'

# obs 1. 
# - zipf distribution matching (with kl regularization target) is able to lift up search advantage
# - soft zipf kl seems to work better than hard zipf kl


In [4]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (select-best per abs SoRL + 1.0 bigram zipf reg + 1.0 traj marg ent) (bulky).gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 200 frames


In [ ]:
# Issue #1. 
# - the figure is off. 
# - very likely, cluster visualizer has abs # starts from 0, but alignment visualizer has abs # starts from 1.
